# Looks To The Sun — interactive demo

Launch LTTS in a temporary Colab runtime. Run the cells from top to bottom.

A T4 runtime is recommended; CPU also works for `SmolLM-135M`.

## 1. Clone the repositories


In [ ]:
!git clone --depth 1 https://github.com/seruva19/LTTS.git
# Analysis modules live in their own repository and are loaded from a
# sibling checkout, so clone it next to LTTS - without it there is
# nothing to attach to a layer.
!git clone --depth 1 https://github.com/seruva19/LTTS-modules.git
%cd LTTS


## 2. Install Python dependencies

Keep Colab's PyTorch build and install the remaining dependencies.

In [ ]:
# Keep Colab's PyTorch and NumPy 2 builds. repeng's NumPy<2 pin is stale,
# so install its already-covered runtime dependencies, then repeng without deps.
!grep -vE '^(--extra-index-url|torch|torchvision|torchaudio|repeng)' requirements.txt > /tmp/ltts_req.txt
!pip install -q -r /tmp/ltts_req.txt
!pip install -q 'gguf>=0.6.0' 'scikit-learn>=1.4.0'
!pip install -q --no-deps repeng==0.4.0
# Cloudflare Quick Tunnels need no account or token. HTTP/2 is used because
# some hosted notebook networks block QUIC/UDP.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
!/tmp/cloudflared --version
print('Python deps and cloudflared installed.')

## 3. Build the frontend

Build the Vue app served by LTTS.

In [ ]:
!npm install --no-audit --no-fund --loglevel=error
!npm run build-client
print('Frontend built into dist/.')

## 4. Start the server and expose it

Start LTTS and print a temporary Cloudflare URL. The URL is public until the runtime or teardown cell stops it.

In [ ]:
import os, subprocess, time, urllib.request
from scripts.utils.quick_tunnel import start_quick_tunnel

PORT = 8000
env = dict(os.environ)
env.update(
    LTTS_HOST='127.0.0.1',
    LTTS_PORT=str(PORT),
    LTTS_SKIP_MODULE_VENVS='1',  # use the base env; skip per-module venvs in the demo
)

server = subprocess.Popen(['python', 'ltts.py'], cwd=os.getcwd(), env=env)

# Wait for the server to come up (importing torch/transformers takes a moment).
health = f'http://127.0.0.1:{PORT}/'
for i in range(90):
    try:
        if urllib.request.urlopen(health, timeout=2).status == 200:
            print(f'Server is up after ~{i*2}s.')
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Server did not start - check the cell output above.')

# Confirm the modules repository was found - an empty list means the
# sibling checkout is missing and nothing can be attached to a layer.
import json as _json
with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/ltts/modules', timeout=10) as _r:
    _modules = _json.load(_r).get('modules', {})
print(f'{len(_modules)} analysis modules loaded.')
if not _modules:
    print('No modules found - re-run cell 1 so LTTS-modules is cloned next to LTTS.')

# Cloudflare Quick Tunnel. The helper reads output on a background thread,
# enforces a real timeout, and reports startup logs on failure.
tunnel, public_url, tunnel_log = start_quick_tunnel(
    PORT, executable='/tmp/cloudflared', timeout=90
)
# Confirm that the public route reaches LTTS before presenting it.
for i in range(30):
    try:
        if urllib.request.urlopen(public_url, timeout=5).status == 200:
            break
    except Exception:
        time.sleep(2)
else:
    tunnel.terminate()
    raise RuntimeError(f'Tunnel URL was assigned but LTTS was not reachable: {public_url}')
print('\n  >>> Open the LTTS GUI here:', public_url, '\n')

## 5. How to test it

Open the printed URL, then:

1. Load `HuggingFaceTB/SmolLM-135M` from **Models -> Load model...**
2. Expand a transformer layer and attach `attn_visualizer` or another module.
3. Enter a prompt in **Model Input** and click **Run**.
4. Check that the module artifact appears in the layer panel.

---
When you're done, stop the tunnel and server with the cell below (or just close the runtime).

In [ ]:
tunnel.terminate()
server.terminate()
print('Tunnel and server stopped.')